# 10 — Movie × Keyword Pairs

Joins the enriched movie dataset (`02_tmdb_movie_enrich`) with the keyword
enrichment table (`00_tmdb_keyword_download`) to produce a simple bipartite edge list:

| column | source |
|---|---|
| `tmdb_movie_id` | `data/tmdb_movies_enriched.csv` |
| `tmdb_keyword_id` | `output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv` |

Movies with no keywords are dropped. Keywords not present in the canonical
enrichment table are also dropped (name not matched).


In [1]:
from pathlib import Path
import pandas as pd

MOVIES_FILE  = Path('data/tmdb_movies_enriched.csv')
KEYWORDS_FILE = Path('output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv')
OUTPUT_FILE  = Path('data/tmdb_movie_keyword_pairs.csv')


## Load

In [2]:
# Load only the columns we need
movies = pd.read_csv(
    MOVIES_FILE,
    usecols=['tmdb_movie_id', 'keywords'],
    low_memory=False,
)

before = len(movies)
movies = movies[movies['keywords'].notna() & (movies['keywords'].str.strip() != '')].copy()
print(f'Movies loaded        : {before:,}')
print(f'With keywords        : {len(movies):,}')
print(f'Dropped (no keywords): {before - len(movies):,}')


Movies loaded        : 1,170,166
With keywords        : 283,881
Dropped (no keywords): 886,285


In [3]:
keywords = pd.read_csv(
    KEYWORDS_FILE,
    usecols=['tmdb_keyword_id', 'name'],
)

# Normalise name for join
keywords['name_norm'] = keywords['name'].str.lower().str.strip()
print(f'Canonical keywords: {len(keywords):,}')


Canonical keywords: 84,842


## Explode and join

In [4]:
# Parse comma-separated keyword names → one row per keyword
movies['kw_list'] = movies['keywords'].str.split(',').apply(
    lambda ks: [k.strip().lower() for k in ks if k.strip()]
)
exploded = (
    movies[['tmdb_movie_id', 'kw_list']]
    .explode('kw_list')
    .rename(columns={'kw_list': 'name_norm'})
    .reset_index(drop=True)
)
print(f'Exploded rows (all keyword mentions): {len(exploded):,}')
print(f'Unique movies:   {exploded["tmdb_movie_id"].nunique():,}')
print(f'Unique keywords: {exploded["name_norm"].nunique():,}')


Exploded rows (all keyword mentions): 957,148
Unique movies:   283,779
Unique keywords: 63,750


In [5]:
# Inner join — drop keyword mentions not in canonical table
pairs = (
    exploded
    .merge(keywords[['tmdb_keyword_id', 'name_norm']], on='name_norm', how='inner')
    [['tmdb_movie_id', 'tmdb_keyword_id']]
    .drop_duplicates()
    .sort_values(['tmdb_movie_id', 'tmdb_keyword_id'])
    .reset_index(drop=True)
)

print(f'Pairs (movie × keyword edges): {len(pairs):,}')
print(f'Unique movies:   {pairs["tmdb_movie_id"].nunique():,}')
print(f'Unique keywords: {pairs["tmdb_keyword_id"].nunique():,}')
pairs.head(10)


Pairs (movie × keyword edges): 928,781
Unique movies:   281,910
Unique keywords: 59,070


,tmdb_movie_id,tmdb_keyword_id
0,2,240
1,2,378
2,2,730
3,2,1563
4,2,13072
5,2,163080
6,2,351252
7,3,1361
8,3,3700
9,3,163080


## Distribution

In [6]:
kw_per_movie  = pairs.groupby('tmdb_movie_id')['tmdb_keyword_id'].count()
mov_per_kw    = pairs.groupby('tmdb_keyword_id')['tmdb_movie_id'].count()

print('Keywords per movie:')
print(kw_per_movie.describe().to_string())
print()
print('Movies per keyword:')
print(mov_per_kw.describe().to_string())


Keywords per movie:
count    281910.000000
mean          3.294601
std           3.382490
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max         102.000000

Movies per keyword:
count    59070.000000
mean        15.723396
std        162.439354
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max      28371.000000


## Keyword usage distribution

How many movies does each keyword appear in?

In [7]:
import plotly.express as px

keywords_meta = pd.read_csv(
    KEYWORDS_FILE,
    usecols=['tmdb_keyword_id', 'name', 'keyword_type', 'is_narrative'],
)

mov_per_kw = (
    pairs.groupby('tmdb_keyword_id')['tmdb_movie_id']
    .count()
    .rename('movie_count')
    .reset_index()
    .merge(keywords_meta, on='tmdb_keyword_id')
)

buckets = [0, 1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, 5000, float('inf')]
labels  = ['1','2','3-5','6-10','11-25','26-50','51-100','101-250','251-500','501-1k','1k-5k','5k+']
mov_per_kw['bucket'] = pd.cut(mov_per_kw['movie_count'], bins=buckets, labels=labels)

dist = mov_per_kw['bucket'].value_counts().reindex(labels).reset_index()
dist.columns = ['movies_per_keyword', 'keyword_count']

hapax     = (mov_per_kw['movie_count'] == 1).sum()
useful    = (mov_per_kw['movie_count'] >= 10).sum()
print(f'Appear in exactly 1 movie : {hapax:,}  ({hapax/len(mov_per_kw)*100:.1f}%)')
print(f'Appear in 10+ movies      : {useful:,}  ({useful/len(mov_per_kw)*100:.1f}%)')

fig = px.bar(
    dist,
    x='movies_per_keyword',
    y='keyword_count',
    text_auto=True,
    title='Keyword usage distribution (movies per keyword)',
    labels={'movies_per_keyword': 'Movies using keyword', 'keyword_count': 'Number of keywords'},
)
fig.update_layout(xaxis_title='Movies using keyword', yaxis_title='Number of keywords')
fig.show()


Appear in exactly 1 movie : 26,541  (44.9%)
Appear in 10+ movies      : 11,521  (19.5%)


In [8]:
print('Top 20 most used keywords:')
top20 = mov_per_kw.nlargest(20, 'movie_count')[['name', 'keyword_type', 'is_narrative', 'movie_count']]
display(top20.reset_index(drop=True))


Top 20 most used keywords:


,name,keyword_type,is_narrative,movie_count
0,short film,theme_or_plot,True,28371
1,woman director,production_meta,False,15458
2,based on novel or book,production_meta,False,6485
3,lgbt,theme_or_plot,True,5109
4,concert,theme_or_plot,True,5061
5,murder,theme_or_plot,True,4829
6,biography,theme_or_plot,True,4029
7,silent film,theme_or_plot,True,3762
8,sports,theme_or_plot,True,3614
9,stand-up comedy,theme_or_plot,True,3603


In [9]:
# Narrative vs non-narrative keyword usage
fig2 = px.histogram(
    mov_per_kw,
    x='movie_count',
    color='is_narrative',
    nbins=50,
    log_x=True,
    log_y=True,
    title='Movie count per keyword (log-log) — narrative vs non-narrative',
    labels={'movie_count': 'Movies per keyword (log)', 'count': 'Keywords (log)'},
    barmode='overlay',
    opacity=0.7,
)
fig2.show()


## Save

In [10]:
pairs.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(pairs):,} pairs → {OUTPUT_FILE}  ({OUTPUT_FILE.stat().st_size / 1_048_576:.1f} MB)')


Saved 928,781 pairs → data/tmdb_movie_keyword_pairs.csv  (11.5 MB)
